# NOTES TO BORA

Currently there are some considerations made for the train/test/val split.

Both test and val are taken from smolsent (Which in total has ~900 entries). This is because it makes no sense to test on gatitos, which may just contain
words that the model hasn't seen, polluting the result.

It DOES mean that training data is heavily skewed toward gatitos, and training data has way more data than test/val. This may be problematic, but there isn't a good way to tell yet.
On the same note, epochs are currently set to 2. It is a small dataset, and therefore more epochs would be good, but I have no idea how long this takes to train. It will be up to you to see if it is feasible.
You may want to play around with the sizes of the dataset to get a better BLEU score. Hopefully it should be fast for you if you have a GPU.


Also, this is currently fine-tuning of a known language (Spanish). We may need to change some things slightly to allow for other languages, but we can discuss it on friday if you don't figure it out.

NOTE: The model used for the first eval run is the base model. The model for the training run is also a copy of the base model, but the new model will be saved to a new folder.

In [ ]:
!pip install sacrebleu

In [6]:
import time
import torch
from torch.utils.data import DataLoader
from datasets import load_dataset, concatenate_datasets
from transformers import MBartForConditionalGeneration, MBart50Tokenizer
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments
from transformers import DataCollatorForSeq2Seq
import evaluate
from tqdm import tqdm
from sklearn.metrics import accuracy_score, f1_score


gatitos = load_dataset("json", data_files="smol/gatitos/en_es.jsonl")["train"]
smolsent = load_dataset("json", data_files="smol/smolsent/en_es.jsonl")["train"]

# Convert 'trgs' list -> single 'trg' string for gatitos
def unify_gatitos(example):
    example["trg"] = example["trgs"][0]
    return example

gatitos = gatitos.map(unify_gatitos)

# Split smolsent into 70/20/10
smolsent_train_test = smolsent.train_test_split(test_size=0.30, seed=42)
smolsent_train = smolsent_train_test["train"]        # 70%
smolsent_test_val = smolsent_train_test["test"]      # remaining 30%

smolsent_val_test = smolsent_test_val.train_test_split(test_size=2/3, seed=42)
smolsent_val = smolsent_val_test["train"]            # 10%
smolsent_test = smolsent_val_test["test"]           # 20%

# Add gatitos to smolsent train
train_dataset = concatenate_datasets([gatitos, smolsent_train])
val_dataset = smolsent_val
test_dataset = smolsent_test

print("Train size:", len(train_dataset))
print("Validation size:", len(val_dataset))
print("Test size:", len(test_dataset))
print("Columns:", train_dataset.column_names)


Train size: 4595
Validation size: 86
Test size: 173
Columns: ['src', 'trgs', 'sl', 'tl', 'is_source_orig', 'trg', 'id', 'is_src_orig']


In [5]:
# Load model
tokenizer = MBart50Tokenizer.from_pretrained("facebook/mbart-large-50-many-to-many-mmt")
model = MBartForConditionalGeneration.from_pretrained("facebook/mbart-large-50-many-to-many-mmt")
model.eval()

# Detect GPU or use CPU
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
print("Using device:", device)
model.to(device)

tokenizer.src_lang = "en_XX"
tokenizer.tgt_lang = "es_XX"

# data loader
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

# Check what is in the data loader?
score = 0
for batch in test_loader:
  print(batch['src'])
  print(batch['trg'])
  if score > 3:
     break
  else:
    score += 1

def evaluate_translation_model(model, tokenizer, test_loader):
    metric = evaluate.load("sacrebleu")

    preds = []
    refs  = []

    for batch in tqdm(test_loader):
        src_texts = batch["src"]
        tgt_texts = batch["trg"]

        # tokenize input
        inputs = tokenizer(src_texts, return_tensors="pt", truncation=True, padding=True).to(model.device)

        # generate translations
        with torch.no_grad():
            generated_ids = model.generate(
                **inputs,
                forced_bos_token_id=tokenizer.lang_code_to_id["es_XX"],
                max_length=128
            )

        # decode output IDs -> text
        decoded_preds = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)
        decoded_preds = [pred.strip() for pred in decoded_preds]

        preds.extend(decoded_preds)
        refs.extend([[t.strip()] for t in tgt_texts])  # sacrebleu expects list of lists

    # compute BLEU
    bleu = metric.compute(predictions=preds, references=refs)
    print(f"BLEU score: {bleu['score']:.2f}")

    return preds, refs, bleu

preds, refs, bleu = evaluate_translation_model(model, tokenizer, test_loader)

Using device: cpu
["You're more likely to fit in at the gym wearing your yoga outfit than trying to impress the high-fashion crowd.", 'Oh no, is everything alright, Latasha replied in a concerned tone?', 'Of course the seedlings always insist on growing either right at the front of the border, or in the vegetable patch.', 'Lucky for him, he’s a brilliant botanist who comes up with clever (sometimes disgusting) ways to harvest food.', 'These worries have shifted the landscape of contemporary childhood in a diversity of ways.', 'Bug where welcome screen link would show blank in admin menu: is it fixed or should I fix it?', 'They weren’t too eager to go fight in a foreign land and lose their own lives.', 'Employee expectations and voices were heard and now we have the courage to move forward with trust.', 'Increasing habitat destruction has seen gray wolf populations shrink in size.', 'As someone who frequently uses lists to manage anxiety and set priorities, I found the general idea attr

100%|██████████| 11/11 [08:26<00:00, 46.01s/it]

BLEU score: 29.05


In [7]:
# Load model
tokenizer = MBart50Tokenizer.from_pretrained("facebook/mbart-large-50-many-to-many-mmt")
model = MBartForConditionalGeneration.from_pretrained("facebook/mbart-large-50-many-to-many-mmt")

tokenizer.src_lang = "en_XX"
tokenizer.tgt_lang = "es_XX"

# Detect GPU or use CPU
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
print("Using device:", device)

model.to(device)

def preprocess_function(batch):
    # batch["src"] = English sentences
    # batch["trg"] = Spanish sentences
    
    # Tokenize source (English)
    model_inputs = tokenizer(
        batch["src"],
        max_length=128,
        padding="max_length",
        truncation=True
    )
    
    # Tokenize target (Spanish)
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            batch["trg"],
            max_length=128,
            padding="max_length",
            truncation=True
        )
    
    # mBART expects 'labels' in the model inputs
    model_inputs["labels"] = labels["input_ids"]
    
    return model_inputs

# tokenize everything
tokenized_train = train_dataset.map(preprocess_function, batched=True, remove_columns=train_dataset.column_names)
tokenized_val   = val_dataset.map(preprocess_function, batched=True, remove_columns=val_dataset.column_names)

# Collator for some padding
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

# Args
output_dir_name = "finetuned-mBart-ES"
training_args = Seq2SeqTrainingArguments(
    output_dir="finetuned-mBart-ES",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2, #Can try more epochs
    weight_decay=0.01,
    save_strategy="epoch",
    predict_with_generate=True,
    logging_strategy="steps",
    logging_steps=20,
    report_to="none"
)


# Define the trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    tokenizer=tokenizer,
    data_collator=data_collator,
)

# Train!
trainer.train()

preds, refs, bleu = evaluate_translation_model(model, tokenizer, test_loader)


Using device: cpu


Map:   0%|          | 0/4595 [00:00<?, ? examples/s]c:\Users\mhusu\miniconda3\envs\nlp\Lib\site-packages\transformers\tokenization_utils_base.py:4169: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(
Map: 100%|██████████| 86/86 [00:00<00:00, 1606.95 examples/s]
C:\Users\mhusu\AppData\Local\Temp\ipykernel_9012\371091405.py:65: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(
c:\Users\mhusu\miniconda3\envs\nlp\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss


KeyboardInterrupt: 